In [ ]:
# RAG ingestion: load documents from knowledge base ---> create golden dataset ---> chunking ---> Chunks validation class ---> Result class ---> embedding and vector database creation ---> EDA
# EDA: bivaraite analysis
import os
import sys
import json
import re
import numpy as np
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from pydantic import BaseModel, field_validator
import pickle
from rank_bm25 import BM25Okapi
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split
# Add the utils folder to Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
pricer_path = os.path.join(notebook_dir, "../utils")
sys.path.insert(0, pricer_path)
# Class Item to validate the returned data and has lots of helper functions
from pricer.items import Item
from sklearn.manifold import TSNE
from plotly import graph_objects as go

In [2]:
# Load environment variables in a file called .env
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
# Check the keys
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")  

OpenAI API Key exists and begins sk-proj-


In [3]:
# Connect to OpenAI client library
openai = OpenAI()

In [4]:
# Configurations
DB_NAME = "preprocessed_db"
collection_name = "docs"
GENERATOR_MODEL = "gpt-5-nano"
EMBEDDING_MODEL = "text-embedding-3-large"
BATCH_SIZE = 500
RAG_CHUNKS = Path("RAG_chunks")
KNOWN_CATEGORIES = {"appliances", "electronics", "home and kitchen", "all beauty"}
SKU_PATTERN = re.compile(r"\b[A-Z0-9][A-Z0-9\-]{4,}\b")  # alphanumeric part/model numbers
GOLDEN_FILE = (
    Path("RAG_eval")
    / "preliminary_golden_dataset.json"
)
BM25_PATH = Path(DB_NAME) / "bm25_index.pkl"
RAG_CHUNKS.mkdir(parents=True, exist_ok=True)
GOLDEN_FILE.parent.mkdir(parents=True, exist_ok=True)

In [5]:
# Invistigate path of the vector database
DB_NAME = str(Path(notebook_dir) / "preprocessed_db")
print("Vector DB path:", DB_NAME)


Vector DB path: c:\Users\EGShaaraOm\OneDrive - NESTLE\Attachments\Work\Software Solutions\Projects\LLMs_Projects\Best_Deal_Gen_AI\notebooks\01_data_pipeline\preprocessed_db


In [6]:
# Validation classes
# Result class
class Result(BaseModel):
    page_content: str
    metadata: dict

In [7]:
class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")
    def as_result(self, document):
        metadata =  {
            "product_id": document["product_id"],
            "source": document["source"],
            "type": document["type"],
            "title": document["title"],
            "price": document["price"]
        }
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)

In [8]:
# Chunks validation class
class Chunks(BaseModel):
    chunks: list[Chunk]

In [9]:
# Data-quality validation class.

class ChunkValidation(BaseModel):
    """Validates a single chunk's page_content + metadata before embedding."""

    page_content: str
    product_id: int
    type: str
    title: str
    price: float
    source: str

    @field_validator("page_content")
    @classmethod
    def content_not_trivial(cls, v):
        v = v.strip()
        if len(v) < 20:
            raise ValueError(f"page_content too short ({len(v)} chars) - likely bad scrape")
        if len(v) > 8000:
            raise ValueError(f"page_content too long ({len(v)} chars) - needs real chunking, not a single chunk")
        return v

    @field_validator("type")
    @classmethod
    def category_known(cls, v):
        if v not in KNOWN_CATEGORIES:
            raise ValueError(f"unknown category: {v!r}")
        return v

    @field_validator("price")
    @classmethod
    def price_sane(cls, v):
        if v <= 0 or v > 100_000:
            raise ValueError(f"suspicious price: {v}")
        return v

    @field_validator("title")
    @classmethod
    def title_not_empty(cls, v):
        if not v.strip():
            raise ValueError("empty title")
        return v


def extract_sku_codes(text: str) -> list[str]:
    """Pull likely SKU / part / model numbers out of a title (for hybrid search)."""
    return sorted(set(SKU_PATTERN.findall(text.upper())))


In [10]:
# Load documents from knowledge base
def fetch_documents():
    # Download the batches dataset from Hugging Face 
    username = "omarshaarawy11"
    dataset = f"{username}/best_deal_03_batches"
    train, val, test = Item.from_hub(dataset)
    # We work with only training and validation data
    items = train + val
    print(f"Loaded {len(items):,} items")
    print(f"Loaded {len(train):,} original training items")
    print(f"Loaded {len(val):,} original validation items")
    print(f"Loaded {len(test):,} original test items")
    # Convert into dataframe
    df = pd.DataFrame(
        [
            {
                "product_id": idx,
                "title": item.title,
                "category": item.category,
                "price": item.price,
                "summary": item.summary
            }
            for idx, item in enumerate(items)
        ]
    )
    print(f"\nTotal products: {len(df):,}")

    print("\nCategory distribution:")
    print(df["category"].value_counts())

    return df

In [11]:
documents = fetch_documents()

Loaded 18,900 items
Loaded 16,800 original training items
Loaded 2,100 original validation items
Loaded 2,191 original test items

Total products: 18,900

Category distribution:
category
appliances          6544
electronics         5967
home and kitchen    5426
all beauty           963
Name: count, dtype: int64


In [12]:
# Invistigate header
documents.head()

,product_id,title,category,price,summary
0,0,Lurrose 100Pcs Full Cover Fake Toenails Artifi...,all beauty,6.99,Title: Lurrose 100pcs Full Cover Fake Toenails...
1,1,"BioMiracle StarDust Pixie Bubble Mask, Clarify...",all beauty,5.99,Title: BioMiracle Green Tea & Apple Carbonated...
2,2,VIROCHEMISTRY Pheromones For Women (Elixir) - ...,all beauty,29.80,Title: ViroChemistry Women's Pheromone Perfume...
3,3,"Garnier Fructis Color Sealer, Instant, Lightwe...",all beauty,24.00,Title: Garnier Fructis Color Sealer Leave-In\n...
4,4,"Enjoy VOLUMIZING ELIXIR, Style (with Sleek Ste...",all beauty,22.49,Title: Enjoy ELIXIR Style with Steel Tail Comb...


In [13]:
# Create golden dataset
# The Golden Dataset represents the category distribution of the complete dataset
GOLDEN_SIZE = 2100
kb_df, golden_df = train_test_split(
    documents,
    test_size=GOLDEN_SIZE,
    stratify=documents["category"],
    random_state=42,
)

kb_df = kb_df.reset_index(drop=True)
golden_df = golden_df.reset_index(drop=True)

print(f"Total products: {len(documents):,}")
print(f"Knowledge Base products: {len(kb_df):,}")
print(f"Golden Dataset products: {len(golden_df):,}")

Total products: 18,900
Knowledge Base products: 16,800
Golden Dataset products: 2,100


In [14]:
print("Full Dataset Distribution:")
print(
    documents["category"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nGolden Dataset Distribution:")
print(
    golden_df["category"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nKnowledge Base Distribution:")
print(
    kb_df["category"]
    .value_counts(normalize=True)
    .sort_index()
)

Full Dataset Distribution:
category
all beauty          0.050952
appliances          0.346243
electronics         0.315714
home and kitchen    0.287090
Name: proportion, dtype: float64

Golden Dataset Distribution:
category
all beauty          0.050952
appliances          0.346190
electronics         0.315714
home and kitchen    0.287143
Name: proportion, dtype: float64

Knowledge Base Distribution:
category
all beauty          0.050952
appliances          0.346250
electronics         0.315714
home and kitchen    0.287083
Name: proportion, dtype: float64


In [15]:
# Make sure no repeated products between the knowledge base and golden dataset
kb_ids = set(kb_df["product_id"])
golden_ids = set(golden_df["product_id"])
print(f"KB products: {len(kb_ids):,}")
print(f"Golden products: {len(golden_ids):,}")
print(f"Overlap: {len(kb_ids.intersection(golden_ids)):,}")
assert len(kb_ids.intersection(golden_ids)) == 0
# Make sure about the total number of products
assert len(kb_ids) + len(golden_ids) == len(documents)

KB products: 16,800
Golden products: 2,100
Overlap: 0


In [16]:
golden_df.to_json(
    GOLDEN_FILE,
    orient="records",
    indent=2,
    force_ascii=False,
)
print(f"Golden Dataset saved to {GOLDEN_FILE.resolve()}")

Golden Dataset saved to C:\Users\EGShaaraOm\OneDrive - NESTLE\Attachments\Work\Software Solutions\Projects\LLMs_Projects\Best_Deal_Gen_AI\notebooks\01_data_pipeline\RAG_eval\preliminary_golden_dataset.json


In [17]:
# Rebuild documents from dataframe
def build_documents(df):

    documents = []

    for _, row in df.iterrows():

        text = f"""
Title: {row['title']}

Category: {row['category']}

Description: {row['summary']}
""".strip()

        documents.append(
            {
                "product_id": int(row["product_id"]),
                "type": row["category"],
                "source": f"product_{row['product_id']}",
                "title": row["title"],
                "text": text,
                "price": float(row["price"]),
            }
        )

    print(f"Built {len(documents):,} documents")

    return documents

In [18]:
kb_documents = build_documents(kb_df)

Built 16,800 documents


In [19]:
def create_chunks(documents):

    chunks = []

    for document in documents:

        metadata = {
            "product_id": document["product_id"],
            "source": document["source"],
            "type": document["type"],
            "title": document["title"],
            "price": document["price"],
        }

        chunk = Result(
            page_content=(
                document["title"]
                + "\n\n"
                + document["text"]
            ),
            metadata=metadata,
        )

        chunks.append(chunk)

    print(
        f"Created {len(chunks):,} chunks "
        f"from {len(documents):,} documents"
    )

    return chunks

In [20]:
chunks = create_chunks(kb_documents)

Created 16,800 chunks from 16,800 documents


In [21]:
# NEW: Run validation over every document, and enrich chunk metadata with
# extracted SKU/model-number codes (used later for hybrid/keyword search).
# Anything that fails validation is reported and dropped, not silently kept.
valid_chunks = []
rejected = []

for document, chunk in zip(kb_documents, chunks):
    try:
        ChunkValidation(
            page_content=chunk.page_content,
            product_id=document["product_id"],
            type=document["type"],
            title=document["title"],
            price=document["price"],
            source=document["source"],
        )
    except Exception as e:
        rejected.append((document["product_id"], str(e)))
        continue

    # Enrich metadata with SKU codes pulled from the title, for hybrid search later
    # Chroma metadata values must be flat scalars (no lists), so we join the codes
    chunk.metadata["sku_codes"] = " ".join(extract_sku_codes(document["title"]))
    valid_chunks.append(chunk)

print(f"Valid chunks: {len(valid_chunks):,} / {len(chunks):,}")
if rejected:
    print(f"Rejected: {len(rejected)}")
    for pid, reason in rejected[:10]:
        print(f"  product_id={pid}: {reason}")

chunks = valid_chunks

Valid chunks: 16,800 / 16,800


In [22]:
# Save chunks to JSON file
chunks_file = RAG_CHUNKS / "kb_chunks.json"
with open(chunks_file, "w", encoding="utf-8") as f:
    json.dump([chunk.model_dump() for chunk in chunks], f, indent=2, ensure_ascii=False)

In [23]:
# Load chunks from JSON file
with open(chunks_file, "r", encoding="utf-8") as f:
    chunks = [Result(**chunk) for chunk in json.load(f)]

In [24]:
# Embedding and vector database creation
def create_embeddings(chunks):

    chroma = PersistentClient(
        path=DB_NAME
    )

    if collection_name in [
        c.name
        for c in chroma.list_collections()
    ]:
        chroma.delete_collection(
            collection_name
        )

    collection = chroma.get_or_create_collection(
        collection_name
    )

    BATCH_SIZE = 100

    total_embeddings = 0

    with tqdm(
        total=len(chunks),
        desc="Creating OpenAI embeddings",
        dynamic_ncols=True,
    ) as progress:

        for start in range(
            0,
            len(chunks),
            BATCH_SIZE
        ):

            batch = chunks[
                start:start + BATCH_SIZE
            ]

            texts = [
                chunk.page_content
                for chunk in batch
            ]

            response = openai.embeddings.create(
                model=EMBEDDING_MODEL,
                input=texts,
            )

            vectors = [
                item.embedding
                for item in response.data
            ]

            ids = [
                str(start + i)
                for i in range(len(batch))
            ]

            metadatas = [
                chunk.metadata
                for chunk in batch
            ]

            collection.add(
                ids=ids,
                embeddings=vectors,
                documents=texts,
                metadatas=metadatas,
            )

            total_embeddings += len(batch)

            progress.update(
                len(batch)
            )

    print(
        f"Vectorstore created with "
        f"{total_embeddings:,} chunks "
        f"and {collection.count():,} embeddings"
    )

In [25]:
create_embeddings(chunks)
print("Ingestion complete")

Creating OpenAI embeddings: 100%|██████████| 16800/16800 [03:44<00:00, 74.71it/s]

Vectorstore created with 16,800 chunks and 16,800 embeddings
Ingestion complete


In [26]:
# Build and persist a BM25 sparse index over the same chunks used for dense embeddings.
# This is the sparse half of hybrid search 
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

def build_bm25_index(chunks):
    corpus_tokens = [tokenize(chunk.page_content + " " + " ".join(chunk.metadata.get("sku_codes", []))) for chunk in chunks]
    bm25 = BM25Okapi(corpus_tokens)

    payload = {
        "corpus_tokens": corpus_tokens,
        "ids": [str(i) for i in range(len(chunks))],
        "metadatas": [chunk.metadata for chunk in chunks],
        "documents": [chunk.page_content for chunk in chunks],
    }

    # Save bm25 index to disk for later retrieval 
    with open(BM25_PATH, "wb") as f:
        pickle.dump(payload, f)

    print(f"BM25 index built over {len(chunks):,} chunks, saved to {BM25_PATH.resolve()}")
    return bm25

bm25_index = build_bm25_index(chunks)


BM25 index built over 16,800 chunks, saved to C:\Users\EGShaaraOm\OneDrive - NESTLE\Attachments\Work\Software Solutions\Projects\LLMs_Projects\Best_Deal_Gen_AI\notebooks\01_data_pipeline\preprocessed_db\bm25_index.pkl


In [27]:
# Dig deeper in vector database created
# Access the persisted Chroma vector database
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_collection(name=collection_name)
count = collection.count()
if count == 0:
    print(f"Collection '{collection_name}' is empty.")
else:
    result = collection.get(
        limit=1,
        include=["embeddings", "documents", "metadatas"],
    )
    sample_embedding = result["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"Database path: {Path(DB_NAME).resolve()}")
    print(f"Collection: {collection_name}")
    print(f"There are {count:,} vectors with {dimensions:,} dimensions")
    print(f"Sample metadata: {result['metadatas'][0]}")
    print(f"Sample document: {result['documents'][0][:200]}...")

Database path: C:\Users\EGShaaraOm\OneDrive - NESTLE\Attachments\Work\Software Solutions\Projects\LLMs_Projects\Best_Deal_Gen_AI\notebooks\01_data_pipeline\preprocessed_db
Collection: docs
There are 16,800 vectors with 3,072 dimensions
Sample metadata: {'product_id': 11699, 'source': 'product_11699', 'title': '2 Pack Air Filter Factory Compatible Replacement for Broan Nutone S99010430-002, 99010430-002, 4512880 Range Hood Aluminum Grease Filters', 'price': 47.97, 'type': 'appliances', 'sku_codes': '4512880 99010430-002 ALUMINUM BROAN COMPATIBLE FACTORY FILTER FILTERS GREASE NUTONE RANGE REPLACEMENT S99010430-002'}
Sample document: 2 Pack Air Filter Factory Compatible Replacement for Broan Nutone S99010430-002, 99010430-002, 4512880 Range Hood Aluminum Grease Filters

Title: 2 Pack Air Filter Factory Compatible Replacement for B...


In [28]:
# EDA
# Bivariate analysis
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
category = [metadata['type'] for metadata in metadatas]
# We map colors with each of categories
colors = [['blue', 'green', 'red', 'orange'][['all beauty', 'appliances', 'electronics', 'home and kitchen'].index(t)] for t in category]

In [29]:
# 2D visual 
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(category, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [31]:
# 3D visual 
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(category, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()